### Write a CUDA Program for :Addition of two large vectors and Matrix Multiplication using CUDA C

In [1]:
#Make sure to: runtime->change runtime type-> GPU->Save
!pip install git+https://github.com/afnan47/cuda.git
%load_ext nvcc_plugin
#Loads CUDA compiler plugin.
#CUDA uses GPU for programming instead of using only CPU. CUDA stands for:
#Compute Unified Device Architecture. Normal programs run on CPU which has
#few powerful cores for general tasks, OS, browsers etc
#GPU has thousands of smaller cores for repeated tasks, matrix operations
#GPU launches thousands of threads. Each thread handles one element paralely

  Cloning https://github.com/afnan47/cuda.git to /tmp/pip-req-build-bttny5vs
  Running command git clone --filter=blob:none --quiet https://github.com/afnan47/cuda.git /tmp/pip-req-build-bttny5vs
  Resolved https://github.com/afnan47/cuda.git to commit aac710a35f52bb78ab34d2e52517237941399eff
  Preparing metadata (setup.py) ... done
directory /content/src already exists
Out bin /content/result.out


In [3]:
%%cu
//“Treat this cell as CUDA C++ code.”

#include <iostream>
using namespace std;

__global__ //This function will run on the GPU. this type of function is called
//kernel function
void add(int* A, int* B, int* C, int size) {
    // * means they are pointers to arrays A,B,C
    //pointers are used Because arrays stored in memory. GPU functions need
    //memory addresses, not full copied arrays.
    //Pointers store address of data in memory
    //size=size of vector arrays

    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    //CUDA structure: grid(blocks(threads))
    //thread is 1 worker doing 1 task. a group of threads is block.
    //tid=thread id. identify each thread globally by id. we find this here.
    //blockIdx.x= current block number(block 0, 1, ...)
    //blockDim.x= number of threads inside one block
    //threadIdx.x= thread number INSIDE current block

    //'blockIdx.x * blockDim.x' = how many threads came before current block
    //Then: '+ threadIdx.x' adds thread position inside current block.
    //Final result: unique global thread ID tid
    //if blockDim.x = 256, current block (blockIdx.x) = 2 and
    //Current thread (threadIdx.x) = 5, then tid = 2 * 256 + 5 = 517
    //needed as thousands of GPU threads run simultaneously.
    //Each thread must know which vector element belongs to me


    if (tid < size) { //Prevents invalid memory access due to extra threads
        C[tid] = A[tid] + B[tid]; //main addition, multple thread (tid) run in parallel
    }
}

void initialize(int* vector, int size) {
    for (int i = 0; i < size; i++) {
        vector[i] = rand() % 10; //Fills vector arrays with random values (0-9)
        //rand() generates random number between 0 and 9 (due to %10)
    }
}

//print vector array
void print(int* vector, int size) {
    for (int i = 0; i < size; i++) {
        cout << vector[i] << " ";
    }
    cout << endl;
}

int main() {

    int N = 4; //vector size =4
    int *A, *B, *C; //pointers for vector arrays

    int vectorSize = N; //vector size=n=4
    size_t vectorBytes = vectorSize * sizeof(int);
    //Calculates memory size in bytes.

    A = new int[vectorSize]; //vector arrays created. cpu memory allocated as it is cpu function
    B = new int[vectorSize];
    C = new int[vectorSize];
    //a is both pointer and vector

    initialize(A, vectorSize); //Fills vectors with random numbers.
    initialize(B, vectorSize);

    cout << "Vector A: ";
    print(A, N);

    cout << "Vector B: ";
    print(B, N);

    int *X, *Y, *Z;
    //a,b,c are cpu pointers Created using new int[vectorSize] (point to cpu memory)
    //x,y,z are gpu pointers created using cudamalloc below (point to gpu memory)
    //two sets of pointers cpu and gpu needed as in CUDA programs,
    //main() runs on CPU, NOT GPU.
    //CPU is the “manager”. it prints output,initializes,launches kernel,handles normal program flow
    //GPU only does specific heavy parallel tasks.

    cudaMalloc(&X, vectorBytes); //gpu memory allocated as it is cuda function which uses gpu
    cudaMalloc(&Y, vectorBytes);
    cudaMalloc(&Z, vectorBytes);


    //CPU and GPU use separate memory spaces.
    //The CPU prepares and manages data, while the GPU performs parallel computation.
    //Therefore, input data must be copied from CPU memory to GPU memory before kernel execution.
    //in simple words: we process all we want in cpu because cpu can process easily and gpu cant
    //then we copy processed arrays from cpu to gpu to perform repetitive tasks on it
    //which gpu can perform easily and cpu cant

    cudaMemcpy(X, A, vectorBytes, cudaMemcpyHostToDevice); //copy cpu A to gpu X
    cudaMemcpy(Y, B, vectorBytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256; //1 block = 256 gpu threads
    int blocksPerGrid =
        (N + threadsPerBlock - 1) / threadsPerBlock;
    //Ensures enough threads for all elements.

    add<<<blocksPerGrid, threadsPerBlock>>>(X, Y, Z, N);
    //earlier we wrote global add so add=GPU function that performs vector addition.
    //arrows mean CUDA kernel launch syntax. tells cuda to run function on gpu
    //inside the arrows we give how much size or threads should be given to function
    //<<<4,256>>> = 4 blocks, 256 threads per block = 1024 GPU threads
    // x,y,z,n given as parameters to add function
    //X,y,z=gpu pointers to 1st,2nd,3rd vectors. n=size of vectors


    cudaMemcpy(C, Z, vectorBytes, cudaMemcpyDeviceToHost);
    //result vector Z copied to C from gpu to cpu.
    //back to cpu as processing done.

    cout << "Addition: ";
    print(C, N); //print final addition vector

    delete[] A; //free cpu memory
    delete[] B;
    delete[] C;

    cudaFree(X); //free gpu memory
    cudaFree(Y);
    cudaFree(Z);

    return 0;
}

Vector A: 3 6 7 5 
Vector B: 3 5 6 2 
Addition: 6 11 13 7 

